# Trabajo Práctico — Series Temporales

## 04 — Estacionariedad

**Objetivo de este notebook:** determinar formalmente, con tests de raíz unitaria (no a ojo), el orden de diferenciación regular (`d`) y estacional (`D`, período 24h) necesario para que cada una de las tres series sea estacionaria. Esto es lo que va a alimentar directamente los órdenes `(p,d,q)(P,D,Q,24)` del SARIMA en `05_sarima.ipynb`.

Se usan tres tests, con hipótesis nula opuesta entre los dos primeros y el tercero — hay que leerlos con cuidado:

- **ADF** (Augmented Dickey-Fuller): H0 = la serie **no** es estacionaria (tiene raíz unitaria). Rechazar H0 (p-valor bajo) = evidencia de estacionariedad.
- **KPSS**: H0 = la serie **sí** es estacionaria. Rechazar H0 (p-valor bajo) = evidencia de **no** estacionariedad. Es decir, se lee al revés que ADF.
- **Phillips-Perron**: misma lógica que ADF (no paramétrico, corrige autocorrelación y heterocedasticidad en el término de error), como test adicional de robustez.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

datos = pd.read_csv(
    PROJECT_ROOT / "data/processed/serie_california.csv",
    index_col="datetime_utc",
    parse_dates=True,
)
print(f"Filas: {len(datos):,} | Rango: {datos.index.min()} -> {datos.index.max()}")

Filas: 239,078 | Rango: 1998-01-01 08:00:00+00:00 -> 2025-08-27 09:00:00+00:00


## 1. Funciones auxiliares

Wrappers estándar para imprimir cada test de forma legible, y una función que corre los tres tests juntos sobre una serie y da un veredicto combinado.

In [2]:
def test_adf(serie, nombre, alpha=0.05):
    resultado = adfuller(serie.dropna(), autolag="AIC")
    pvalor = resultado[1]
    print(f"[ADF] {nombre}: estadístico={resultado[0]:.3f}  p-valor={pvalor:.4f}  "
          f"-> {'estacionaria' if pvalor < alpha else 'NO estacionaria'}")
    return pvalor < alpha


def test_kpss(serie, nombre, alpha=0.05, regression="c"):
    resultado = kpss(serie.dropna(), regression=regression, nlags="auto")
    pvalor = resultado[1]
    print(f"[KPSS] {nombre}: estadístico={resultado[0]:.3f}  p-valor={pvalor:.4f}  "
          f"-> {'estacionaria' if pvalor > alpha else 'NO estacionaria'}")
    return pvalor > alpha


def test_pp(serie, nombre, alpha=0.05):
    resultado = PhillipsPerron(serie.dropna())
    pvalor = resultado.pvalue
    print(f"[PP]   {nombre}: estadístico={resultado.stat:.3f}  p-valor={pvalor:.4f}  "
          f"-> {'estacionaria' if pvalor < alpha else 'NO estacionaria'}")
    return pvalor < alpha


def veredicto_estacionariedad(serie, nombre, alpha=0.05):
    print(f"--- {nombre} ---")
    adf_ok = test_adf(serie, nombre, alpha)
    kpss_ok = test_kpss(serie, nombre, alpha)
    pp_ok = test_pp(serie, nombre, alpha)
    votos = sum([adf_ok, kpss_ok, pp_ok])
    conclusion = "ESTACIONARIA" if votos >= 2 else "NO estacionaria"
    print(f"Conclusión ({votos}/3 tests a favor de estacionariedad): {conclusion}\n")
    return votos >= 2

## 2. Series en nivel (sin diferenciar)

Se espera que ninguna de las tres sea estacionaria en nivel: ozono y radiación tienen un ciclo diario fuerte (no es ruido alrededor de una media constante, la media *depende* de la hora del día), y temperatura tiene además tendencia de más largo plazo.

In [3]:
resultados_nivel = {}
for col in datos.columns:
    resultados_nivel[col] = veredicto_estacionariedad(datos[col], col)

--- ozono_ppm ---


[ADF] ozono_ppm: estadístico=-22.388  p-valor=0.0000  -> estacionaria
[KPSS] ozono_ppm: estadístico=24.482  p-valor=0.0100  -> NO estacionaria


C:\Users\leonardo.gastaldo\AppData\Local\Temp\ipykernel_18080\1803190774.py:10: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  resultado = kpss(serie.dropna(), regression=regression, nlags="auto")


[PP]   ozono_ppm: estadístico=-77.845  p-valor=0.0000  -> estacionaria
Conclusión (2/3 tests a favor de estacionariedad): ESTACIONARIA

--- temperatura_c ---


[ADF] temperatura_c: estadístico=-18.980  p-valor=0.0000  -> estacionaria
[KPSS] temperatura_c: estadístico=1.307  p-valor=0.0100  -> NO estacionaria


...

[PP]   temperatura_c: estadístico=-60.714  p-valor=0.0000  -> estacionaria
Conclusión (2/3 tests a favor de estacionariedad): ESTACIONARIA

--- radiacion_ghi_wm2 ---


[ADF] radiacion_ghi_wm2: estadístico=-18.732  p-valor=0.0000  -> estacionaria


C:\Users\leonardo.gastaldo\AppData\Local\Temp\ipykernel_18080\1803190774.py:10: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  resultado = kpss(serie.dropna(), regression=regression, nlags="auto")


[KPSS] radiacion_ghi_wm2: estadístico=0.013  p-valor=0.1000  -> estacionaria


[PP]   radiacion_ghi_wm2: estadístico=-61.170  p-valor=0.0000  -> estacionaria
Conclusión (3/3 tests a favor de estacionariedad): ESTACIONARIA



## 3. Diferenciación estacional (D, período 24h)

Como el ciclo dominante es diario, se prueba primero la diferencia estacional (`.diff(24)`) antes que la regular — a diferencia de los ejemplos de la cátedra (series mensuales/diarias sin un ciclo intradiario tan marcado), acá el primer paso lógico es sacar el ciclo de 24 horas, no una tendencia de largo plazo.

In [4]:
PERIODO_ESTACIONAL = 24

datos_diff_estacional = datos.diff(PERIODO_ESTACIONAL).dropna()

resultados_diff_estacional = {}
for col in datos.columns:
    resultados_diff_estacional[col] = veredicto_estacionariedad(
        datos_diff_estacional[col], f"{col} (diff estacional D=1)"
    )

--- ozono_ppm (diff estacional D=1) ---


[ADF] ozono_ppm (diff estacional D=1): estadístico=-70.078  p-valor=0.0000  -> estacionaria
[KPSS] ozono_ppm (diff estacional D=1): estadístico=0.001  p-valor=0.1000  -> estacionaria


C:\Users\leonardo.gastaldo\AppData\Local\Temp\ipykernel_18080\1803190774.py:10: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  resultado = kpss(serie.dropna(), regression=regression, nlags="auto")


[PP]   ozono_ppm (diff estacional D=1): estadístico=-95.534  p-valor=0.0000  -> estacionaria
Conclusión (3/3 tests a favor de estacionariedad): ESTACIONARIA

--- temperatura_c (diff estacional D=1) ---


[ADF] temperatura_c (diff estacional D=1): estadístico=-67.048  p-valor=0.0000  -> estacionaria
[KPSS] temperatura_c (diff estacional D=1): estadístico=0.002  p-valor=0.1000  -> estacionaria


C:\Users\leonardo.gastaldo\AppData\Local\Temp\ipykernel_18080\1803190774.py:10: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  resultado = kpss(serie.dropna(), regression=regression, nlags="auto")


[PP]   temperatura_c (diff estacional D=1): estadístico=-113.069  p-valor=0.0000  -> estacionaria
Conclusión (3/3 tests a favor de estacionariedad): ESTACIONARIA

--- radiacion_ghi_wm2 (diff estacional D=1) ---


[ADF] radiacion_ghi_wm2 (diff estacional D=1): estadístico=-72.516  p-valor=0.0000  -> estacionaria
[KPSS] radiacion_ghi_wm2 (diff estacional D=1): estadístico=0.001  p-valor=0.1000  -> estacionaria


C:\Users\leonardo.gastaldo\AppData\Local\Temp\ipykernel_18080\1803190774.py:10: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  resultado = kpss(serie.dropna(), regression=regression, nlags="auto")


[PP]   radiacion_ghi_wm2 (diff estacional D=1): estadístico=-104.679  p-valor=0.0000  -> estacionaria
Conclusión (3/3 tests a favor de estacionariedad): ESTACIONARIA



## 4. Diferenciación regular adicional (d), si todavía hace falta

Para las series que sigan sin ser estacionarias después de la diferencia estacional, se aplica además una diferencia regular (`d=1`) sobre la serie ya diferenciada estacionalmente, y se vuelve a testear.

In [5]:
d_por_serie = {}
D_por_serie = {}

for col in datos.columns:
    D_por_serie[col] = 1  # ya se aplicó arriba a las tres

    if resultados_diff_estacional[col]:
        d_por_serie[col] = 0
        print(f"{col}: estacionaria con sólo D=1 (d=0)\n")
    else:
        serie_dd = datos_diff_estacional[col].diff().dropna()
        es_estacionaria = veredicto_estacionariedad(serie_dd, f"{col} (diff estacional + regular)")
        d_por_serie[col] = 1
        if not es_estacionaria:
            print(f"ADVERTENCIA: {col} sigue sin ser estacionaria con D=1,d=1 — revisar antes de pasar a SARIMA.\n")

print("Resumen de órdenes de diferenciación sugeridos:")
for col in datos.columns:
    print(f"  {col}: d={d_por_serie[col]}  D={D_por_serie[col]}  (período estacional=24)")

ozono_ppm: estacionaria con sólo D=1 (d=0)

temperatura_c: estacionaria con sólo D=1 (d=0)

radiacion_ghi_wm2: estacionaria con sólo D=1 (d=0)

Resumen de órdenes de diferenciación sugeridos:
  ozono_ppm: d=0  D=1  (período estacional=24)
  temperatura_c: d=0  D=1  (período estacional=24)
  radiacion_ghi_wm2: d=0  D=1  (período estacional=24)


## 5. Guardado de los órdenes sugeridos

Se guardan `d` y `D` por serie en un JSON simple, para que `05_sarima.ipynb` los lea directamente en vez de tener que volver a correr los tests.

In [6]:
import json

ordenes = {
    col: {"d": d_por_serie[col], "D": D_por_serie[col], "periodo_estacional": PERIODO_ESTACIONAL}
    for col in datos.columns
}

destino = PROJECT_ROOT / "data/processed/ordenes_diferenciacion.json"
destino.write_text(json.dumps(ordenes, indent=2), encoding="utf-8")
print(f"Guardado en {destino}")
print(json.dumps(ordenes, indent=2))

Guardado en C:\Series_AMBA\data\processed\ordenes_diferenciacion.json
{
  "ozono_ppm": {
    "d": 0,
    "D": 1,
    "periodo_estacional": 24
  },
  "temperatura_c": {
    "d": 0,
    "D": 1,
    "periodo_estacional": 24
  },
  "radiacion_ghi_wm2": {
    "d": 0,
    "D": 1,
    "periodo_estacional": 24
  }
}


**Proyecto listo para continuar con `05_sarima.ipynb`** (grid search de `(p,q)(P,Q)` usando los `d`/`D` ya determinados acá, selección por AIC, diagnóstico de residuos y pronóstico).